In [ ]:
!git clone https://github.com/reemi96/AI_Based_Grading_System.git /content/AI_Based_Grading_System

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip uninstall -y torchao
!pip install -q -U transformers peft accelerate bitsandbytes fastapi uvicorn pyngrok

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_model_name = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
lora_path = "/content/AI_Based_Grading_System/model/rubric_codegrader_qwen_lora_600samples_epochs8"

tokenizer = AutoTokenizer.from_pretrained(base_model_name)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

model = PeftModel.from_pretrained(base_model, lora_path)
model.eval()

print("Model loaded successfully")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded successfully


In [ ]:
import re
import torch

def extract_score(text):
    match = re.search(r"(\d+(?:\.\d+)?)\s*/\s*10", text)
    if match:
        return f"{match.group(1)}/10"

    match = re.search(r"(?:العلامة|الدرجة)\s*[:：]?\s*(\d+(?:\.\d+)?)", text)
    if match:
        return f"{match.group(1)}/10"

    return text.strip()

def predict_grade(
    subject: str,
    qid: str,
    question_text: str,
    rubric: str,
    reference_solution: str,
    student_answer: str
):
    prompt = f"""
صحح حل الطالب اعتماداً على السؤال، سلم التصحيح، والحل النموذجي.

المادة: {subject}
رقم السؤال: {qid}

السؤال:
{question_text}

سلم التصحيح:
{rubric}

الحل النموذجي:
{reference_solution}

حل الطالب:
{student_answer}

أعط العلامة النهائية فقط من 10.
""".strip()

    messages = [
        {
            "role": "system",
            "content": "صحح حل الطالب وأعد العلامة النهائية فقط بصيغة رقم/10."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    prediction = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )
    return extract_score(prediction)

In [ ]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn
import threading

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class GradeRequest(BaseModel):
    subject: str
    qid: str
    question_text: str
    rubric: str
    reference_solution: str
    student_answer: str

@app.get("/")
def home():
    return {"status": "running"}

@app.post("/grade")
def grade(request: GradeRequest):
    result = predict_grade(
        subject=request.subject,
        qid=request.qid,
        question_text=request.question_text,
        rubric=request.rubric,
        reference_solution=request.reference_solution,
        student_answer=request.student_answer
    )

    return {
        "predicted_grade": result,
        "raw_response": result
    }

def run_api():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_api)
thread.start()

print("FastAPI is running on port 8000")

FastAPI is running on port 8000


In [ ]:
from google.colab import userdata
from pyngrok import ngrok

NGROK_TOKEN = userdata.get("NGROK_TOKEN")
ngrok.set_auth_token(NGROK_TOKEN)

print("ngrok token loaded successfully.")

INFO:     Started server process [15834]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


ngrok token loaded successfully.


In [ ]:
from pyngrok import ngrok

# أغلق أي نفق قديم
ngrok.kill()

# افتح نفق على المنفذ 8000
public_url = ngrok.connect(8000)

print("Public URL:")
print(public_url.public_url)

Public URL:
https://underhand-squeamish-overbite.ngrok-free.dev
